# This is used to construct the dataset with the desired features to train out model on


In [151]:
# Firstly lets load our inital data

import pandas as pd
import numpy as np
import networkx as nx

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from scipy import sparse

data_dir = Path("Elliptic++ Dataset")
addraddr_edgelist = pd.read_csv(data_dir / "AddrAddr_edgelist.csv")
AddrTx_edgelist = pd.read_csv(data_dir / "AddrTx_edgelist.csv")
TxAddr_edgelist = pd.read_csv(data_dir / "TxAddr_edgelist.csv")
txs_classes = pd.read_csv(data_dir / "txs_classes.csv")
txs_edgelist = pd.read_csv(data_dir / "txs_edgelist.csv")
txs_features = pd.read_csv(data_dir / "txs_features.csv")
wallet_classes = pd.read_csv(data_dir / "wallets_classes.csv")
wallet_features_classes_combined = pd.read_csv(data_dir / "wallets_features_classes_combined.csv")
wallet_features = pd.read_csv(data_dir / "wallets_features.csv")


# Input Count

$input\_count(T) = |\{unique\ sending\ addresses\ participating\ in\ transaction\ T\}|$

In [152]:
input_counts = (
    AddrTx_edgelist
    .groupby("txId")["input_address"]
    .nunique()
    .reset_index(name="input_count")
)

print(input_counts.head())

   txId  input_count
0  1076            2
1  2534            3
2  3181            1
3  3321            1
4  3889            1


# output_count

$output_count(T) = number\ of\ unique\ recieving\ addresses\ for\ tranaction\ T$

In [153]:
output_counts = (
    TxAddr_edgelist
    .groupby("txId")["output_address"]
    .nunique()
    .reset_index(name="output_count")
)

print(output_counts.head())

   txId  output_count
0  1076             2
1  2534             7
2  3181          4906
3  3321             2
4  3889           863


# Input Total

$input_total(T) = \sum_{inputs\ into\ T} BTC\ value$

In [154]:
# TODO fix needs to be the column of input size

input_totals = txs_features[
    ["txId", "num_input_addresses"]
].copy()

input_totals = input_totals.rename(
    columns={"num_input_addresses": "input_total"}
)

print(input_totals.head())


    txId  input_total
0   3321          1.0
1  11108          1.0
2  51816          1.0
3  68869          3.0
4  89273          1.0


In [155]:
print(txs_features.columns[-20:].tolist())

['Aggregate_feature_70', 'Aggregate_feature_71', 'Aggregate_feature_72', 'in_txs_degree', 'out_txs_degree', 'total_BTC', 'fees', 'size', 'num_input_addresses', 'num_output_addresses', 'in_BTC_min', 'in_BTC_max', 'in_BTC_mean', 'in_BTC_median', 'in_BTC_total', 'out_BTC_min', 'out_BTC_max', 'out_BTC_mean', 'out_BTC_median', 'out_BTC_total']


# Already existing columns that I do have to redo lol

In [156]:
in_out_btc_features = txs_features[
    [
        "txId",
        "in_BTC_min",
        "in_BTC_max",
        "in_BTC_mean",
        "in_BTC_median",
        "in_BTC_total",
        "out_BTC_min",
        "out_BTC_max",
        "out_BTC_mean",
        "out_BTC_median",
        "out_BTC_total",
    ]
].copy()

print(in_out_btc_features.head())




    txId  in_BTC_min  in_BTC_max  in_BTC_mean  in_BTC_median  in_BTC_total  \
0   3321    0.534072    0.534072     0.534072       0.534072      0.534072   
1  11108    5.611878    5.611878     5.611878       5.611878      5.611878   
2  51816    0.456608    0.456608     0.456608       0.456608      0.456608   
3  68869    0.308900    8.000000     3.102967       1.000000      9.308900   
4  89273  852.164680  852.164680   852.164680     852.164680    852.164680   

    out_BTC_min  out_BTC_max  out_BTC_mean  out_BTC_median  out_BTC_total  
0  1.668990e-01     0.367074      0.266986        0.266986       0.533972  
1  5.861940e-01     5.025584      2.805889        2.805889       5.611778  
2  2.279902e-01     0.228518      0.228254        0.228254       0.456508  
3  1.229000e+00     8.079800      4.654400        4.654400       9.308800  
4  1.300000e-07    41.264036      0.065016        0.000441     852.164680  


# Input historical data :(

First starting with building a lookup table for each address

In [157]:
tx_to_input_addresses = (
    AddrTx_edgelist
    .groupby("txId")["input_address"]
    .unique()
    .reset_index(name="input_addresses")
)
print(tx_to_input_addresses.head())

   txId                                    input_addresses
0  1076  [18LEv2TV1sg1GkGd6w5FdD3vp1dmiTTcNr, 1LXcdzQpi...
1  2534  [149hEPHkEWH31qaMRz77tFxjk3WrQkT1P1, 1AYHBouG6...
2  3181               [1GX28yLjVWux7ws4UQ9FB4MnLH4UKTPK2z]
3  3321               [18TnVVsNqViRGTNzCjk5dnW4VNxpehywXB]
4  3889               [1LjYX8JCSSCgzUdEHVZi4qcdxzn7YEkr1M]


In [158]:
tx_to_output_addresses = (
    TxAddr_edgelist
    .groupby("txId")["output_address"]
    .unique()
    .reset_index(name="output_addresses")
)
print(tx_to_output_addresses.head())

   txId                                   output_addresses
0  1076  [1DfE5KELWYYhgDsNcvhTQwMUHLtsz32Kj, 3D28Kvhgbb...
1  2534  [1PDf2CkNLa7TkZmQSW3njkkoMHZpU1GVXF, 1MFt5JfB6...
2  3181  [1GKkMWXLHXVVfv9RY1oZDtuFEmo4Xf3m3V, 1D9DWDH1b...
3  3321  [18TnVVsNqViRGTNzCjk5dnW4VNxpehywXB, 1EhgvhK2N...
4  3889  [1DovMcRhEoXpsaG4xod9RnuBZ3K6rtLosR, 19rK6LmWG...


In [159]:
sending_participation = AddrTx_edgelist[
    ["input_address", "txId"]
].copy()

recieving_participation = TxAddr_edgelist[
    ["output_address", "txId"]
].copy()

In [160]:
address_transactions = pd.concat(
    [sending_participation, recieving_participation]
    , ignore_index=True
)
# print the row(s) where txId == 1076
# print(address_transactions[address_transactions["txId"] == 1076])


address_transactions["address"] = (
    address_transactions["input_address"]
    .combine_first(address_transactions["output_address"])
)

address_transactions = address_transactions.drop(columns=["input_address", "output_address"])

address_transactions = (
    address_transactions.drop_duplicates(subset=["txId", "address"])
)

print(address_transactions[address_transactions["txId"] == 1076])


         txId                             address
466078   1076  18LEv2TV1sg1GkGd6w5FdD3vp1dmiTTcNr
466079   1076  1LXcdzQpiWoaAVvRRETdWmS4YXAe7qD3Nq
1300701  1076   1DfE5KELWYYhgDsNcvhTQwMUHLtsz32Kj
1300702  1076  3D28KvhgbbFpYQAgyabVn4aiBhFTobE864


In [161]:
address_to_transactions = (
    address_transactions
    .groupby("address")["txId"]
    .unique()
    .reset_index(name="transactions")
)

print(address_to_transactions.head())

                              address  \
0   111112TykSw72ztDN2WJger4cynzWYC5w   
1   1111DAYXhoxZx2tsRnzimfozo783x1yC2   
2    1111VHuXEzHaRCgXbVwojtaP7Co3QABb   
3  111218KKkh1JJFRHbwM16AwCiVCc4m7he1   
4   1115LWW3xsD9jT9VRY7viCN9S34RVAAuA   

                                        transactions  
0                                         [15329918]  
1  [50030829, 163620145, 97229578, 97231180, 9436...  
2                                          [3725934]  
3                             [390061088, 390060613]  
4                                          [2107102]  


In [162]:
address_transactions = address_transactions.merge(
    txs_features[['txId', 'Time step']],
    on='txId',
    how='left'
)

print(address_transactions.head())

        txId                             address  Time step
0  230325127  14YRXHHof4BY1TVxN5FqYPcEdpmXiYT78a          1
1  230325139  13Lhad3SAmu2vqYg2dxbNcxH7LE77kJu2w          1
2   86875675  1MAQQZn7EHP6J3erXByCciFiVcgS8ZhWqz          1
3  230325147  16zs5SVSyADh5WrLNbZbpRLsBsN5uEzgeK          1
4  230325154  1QJpwtUorBKPGUJkSyrRcBKTAHq4CXrdYh          1


In [163]:
# merge Time step_x and _y

# address_transactions["Time step"] = address_transactions["Time step_x"]
# address_transactions = address_transactions.drop(
#     columns=["Time step_y", "Time step_x"]
# )

# print(address_transactions.head())

KeyError: 'Time step_x'

## Count each addresses activity per time step

In [164]:
address_step_counts = (
    address_transactions.groupby(["address", "Time step"])
    .size()
    .reset_index(name="tx_count_at_step")
)

print(address_step_counts.head())

                             address  Time step  tx_count_at_step
0  111112TykSw72ztDN2WJger4cynzWYC5w         25                 1
1  1111DAYXhoxZx2tsRnzimfozo783x1yC2         25                 1
2  1111DAYXhoxZx2tsRnzimfozo783x1yC2         29                 1
3  1111DAYXhoxZx2tsRnzimfozo783x1yC2         39                 2
4  1111DAYXhoxZx2tsRnzimfozo783x1yC2         43                 2


## Now we can calculate how may transactions happened before each time step

In [165]:
address_step_counts = address_step_counts.sort_values(
    ["address", "Time step"]
)

address_step_counts["prior_tx_count"] = (
    address_step_counts.groupby("address")["tx_count_at_step"]
    .cumsum() - address_step_counts["tx_count_at_step"]
)

print(address_step_counts.head())
print(address_step_counts.columns)

                             address  Time step  tx_count_at_step  \
0  111112TykSw72ztDN2WJger4cynzWYC5w         25                 1   
1  1111DAYXhoxZx2tsRnzimfozo783x1yC2         25                 1   
2  1111DAYXhoxZx2tsRnzimfozo783x1yC2         29                 1   
3  1111DAYXhoxZx2tsRnzimfozo783x1yC2         39                 2   
4  1111DAYXhoxZx2tsRnzimfozo783x1yC2         43                 2   

   prior_tx_count  
0               0  
1               0  
2               1  
3               2  
4               4  
Index(['address', 'Time step', 'tx_count_at_step', 'prior_tx_count'], dtype='str')


### Now we want to attach the to the transaction id for each address

In [166]:
AddrTx_with_time = AddrTx_edgelist.merge(
    txs_features[["txId", "Time step"]],
    on="txId",
    how="left"
)

print(AddrTx_with_time.head())

                        input_address       txId  Time step
0  14YRXHHof4BY1TVxN5FqYPcEdpmXiYT78a  230325127          1
1  13Lhad3SAmu2vqYg2dxbNcxH7LE77kJu2w  230325139          1
2  1MAQQZn7EHP6J3erXByCciFiVcgS8ZhWqz   86875675          1
3  16zs5SVSyADh5WrLNbZbpRLsBsN5uEzgeK  230325147          1
4  1QJpwtUorBKPGUJkSyrRcBKTAHq4CXrdYh  230325154          1


In [167]:
input_address_history = AddrTx_with_time.merge(
    address_step_counts[
        ["address", "Time step", "prior_tx_count"]
    ],
    left_on=["input_address", "Time step"],
    right_on=["address", "Time step"],
    how="left"
)

print(input_address_history.head())

                        input_address       txId  Time step  \
0  14YRXHHof4BY1TVxN5FqYPcEdpmXiYT78a  230325127          1   
1  13Lhad3SAmu2vqYg2dxbNcxH7LE77kJu2w  230325139          1   
2  1MAQQZn7EHP6J3erXByCciFiVcgS8ZhWqz   86875675          1   
3  16zs5SVSyADh5WrLNbZbpRLsBsN5uEzgeK  230325147          1   
4  1QJpwtUorBKPGUJkSyrRcBKTAHq4CXrdYh  230325154          1   

                              address  prior_tx_count  
0  14YRXHHof4BY1TVxN5FqYPcEdpmXiYT78a               0  
1  13Lhad3SAmu2vqYg2dxbNcxH7LE77kJu2w               0  
2  1MAQQZn7EHP6J3erXByCciFiVcgS8ZhWqz               0  
3  16zs5SVSyADh5WrLNbZbpRLsBsN5uEzgeK               0  
4  1QJpwtUorBKPGUJkSyrRcBKTAHq4CXrdYh               0  


# Now to find prior txs average / max / min

In [168]:
input_prior_tx_avg = (
    input_address_history
    .groupby("txId")["prior_tx_count"]
    .mean()
    .reset_index(name="input_prior_tx_avg")
)

print(input_prior_tx_avg.head())

   txId  input_prior_tx_avg
0  1076                 0.0
1  2534                 0.0
2  3181                 8.0
3  3321                 0.0
4  3889                 0.0


In [169]:
input_prior_tx_max = (
    input_address_history
    .groupby("txId")["prior_tx_count"]
    .max()
    .reset_index(name="input_prior_tx_max")
)

print(input_prior_tx_max.head())

   txId  input_prior_tx_max
0  1076                   0
1  2534                   0
2  3181                   8
3  3321                   0
4  3889                   0


In [170]:
input_prior_tx_min = (
    input_address_history
    .groupby("txId")["prior_tx_count"]
    .min()
    .reset_index(name="input_prior_tx_min")
)

print(input_prior_tx_min.head())

   txId  input_prior_tx_min
0  1076                   0
1  2534                   0
2  3181                   8
3  3321                   0
4  3889                   0


### input previous max amount

maximum prior transaction count among that transaction’s sending addresses. How historically active was the single most active sending wallet involved in this transaction?

In [171]:
input_prior_tx_max = (
    input_address_history
    .groupby("txId")["prior_tx_count"]
    .max()
    .reset_index(name="input_prior_tx_max")
)

print(input_prior_tx_max.head())

   txId  input_prior_tx_max
0  1076                   0
1  2534                   0
2  3181                   8
3  3321                   0
4  3889                   0


# Btc sent stuff now

For the input/sending addresses of the current transaction, how much BTC had those addresses sent in earlier transactions?

$BTC_{sent}(T) = BTC_{sent\ before\ T\ by\ A} + BTC_{sent\ before\ T\ by\ B} + \...$

In [172]:
sending_history = AddrTx_with_time.merge(
    txs_features[
        ["txId", "in_BTC_total"]
    ],
    on="txId",
    how="left"
)
print(sending_history.head())

                        input_address       txId  Time step  in_BTC_total
0  14YRXHHof4BY1TVxN5FqYPcEdpmXiYT78a  230325127          1      7.000303
1  13Lhad3SAmu2vqYg2dxbNcxH7LE77kJu2w  230325139          1      5.525902
2  1MAQQZn7EHP6J3erXByCciFiVcgS8ZhWqz   86875675          1     11.811274
3  16zs5SVSyADh5WrLNbZbpRLsBsN5uEzgeK  230325147          1      3.770000
4  1QJpwtUorBKPGUJkSyrRcBKTAHq4CXrdYh  230325154          1      3.200399


In [173]:
current_inputs = AddrTx_with_time[
    ["txId", "input_address", "Time step"]
].copy()

current_inputs = current_inputs.rename(columns={
    "txId": "current_txId",
    "Time step": "current_time_step"
})

historical_sends = sending_history[
    ["txId", "input_address", "Time step", "in_BTC_total"]
].copy()

historical_sends = historical_sends.rename(columns={
    "txId": "historical_txId",
    "Time step": "historical_time_step"
})

print(current_inputs.head())

print(historical_sends.head())

   current_txId                       input_address  current_time_step
0     230325127  14YRXHHof4BY1TVxN5FqYPcEdpmXiYT78a                  1
1     230325139  13Lhad3SAmu2vqYg2dxbNcxH7LE77kJu2w                  1
2      86875675  1MAQQZn7EHP6J3erXByCciFiVcgS8ZhWqz                  1
3     230325147  16zs5SVSyADh5WrLNbZbpRLsBsN5uEzgeK                  1
4     230325154  1QJpwtUorBKPGUJkSyrRcBKTAHq4CXrdYh                  1
   historical_txId                       input_address  historical_time_step  \
0        230325127  14YRXHHof4BY1TVxN5FqYPcEdpmXiYT78a                     1   
1        230325139  13Lhad3SAmu2vqYg2dxbNcxH7LE77kJu2w                     1   
2         86875675  1MAQQZn7EHP6J3erXByCciFiVcgS8ZhWqz                     1   
3        230325147  16zs5SVSyADh5WrLNbZbpRLsBsN5uEzgeK                     1   
4        230325154  1QJpwtUorBKPGUJkSyrRcBKTAHq4CXrdYh                     1   

   in_BTC_total  
0      7.000303  
1      5.525902  
2     11.811274  
3      3.770000  
4  

In [174]:
input_send_matches = current_inputs.merge(
    historical_sends,
    on="input_address",
    how="left"
)
print(input_send_matches.head())

   current_txId                       input_address  current_time_step  \
0     230325127  14YRXHHof4BY1TVxN5FqYPcEdpmXiYT78a                  1   
1     230325139  13Lhad3SAmu2vqYg2dxbNcxH7LE77kJu2w                  1   
2      86875675  1MAQQZn7EHP6J3erXByCciFiVcgS8ZhWqz                  1   
3     230325147  16zs5SVSyADh5WrLNbZbpRLsBsN5uEzgeK                  1   
4     230325154  1QJpwtUorBKPGUJkSyrRcBKTAHq4CXrdYh                  1   

   historical_txId  historical_time_step  in_BTC_total  
0        230325127                     1      7.000303  
1        230325139                     1      5.525902  
2         86875675                     1     11.811274  
3        230325147                     1      3.770000  
4        230325154                     1      3.200399  


In [175]:
input_send_matches = input_send_matches[
    input_send_matches["historical_time_step"]
    < input_send_matches["current_time_step"]
]

print(input_send_matches.head())

        current_txId                       input_address  current_time_step  \
240538     309947457  1Po4J4SNyJuGnMGYJfGTXLEvGgAZKiddr7                  2   
240539     309947457  1Po4J4SNyJuGnMGYJfGTXLEvGgAZKiddr7                  2   
240540     309947457  1Po4J4SNyJuGnMGYJfGTXLEvGgAZKiddr7                  2   
240541     309947457  1Po4J4SNyJuGnMGYJfGTXLEvGgAZKiddr7                  2   
240542     309947457  1Po4J4SNyJuGnMGYJfGTXLEvGgAZKiddr7                  2   

        historical_txId  historical_time_step  in_BTC_total  
240538        232349075                     1      0.229163  
240539        232349072                     1      2.131743  
240540         27323627                     1      3.596236  
240541        121435080                     1      1.610202  
240542         40587631                     1      0.301813  


In [176]:
# Remove dupes

unique_prior_sends = input_send_matches.drop_duplicates(
    subset=["current_txId", "historical_txId"]
)

In [177]:
input_btc_sent = (
    unique_prior_sends
    .groupby("current_txId")["in_BTC_total"]
    .sum()
    .reset_index(name="input_btc_sent")
)

input_btc_sent = input_btc_sent.rename(
    columns={"current_txId": "txId"}
)

print(input_btc_sent.head())

   txId  input_btc_sent
0  3181     2325.189831
1  6418      627.139089
2  7952     2559.542194
3  9363   244297.263276
4  9466    10124.186381


# Make mean/max/min for sending wallet prior transactions


In [178]:
input_btc_stats = (
    unique_prior_sends
    .groupby("current_txId")["in_BTC_total"]
    .agg(
        input_btc_avg="mean",
        input_btc_min="min",
        input_btc_max="max"
    )
    .reset_index()
    .rename(columns={"current_txId": "txId"})
)

print(input_btc_stats.head())

   txId  input_btc_avg  input_btc_min  input_btc_max
0  3181     581.297458     293.037124     885.427232
1  6418     209.046363      39.573100     319.700243
2  7952     511.908439     234.352363     885.427232
3  9363     534.567316       1.909120    3430.321674
4  9466     103.308024       3.248224     956.069135


# Now Repeat for the output addresses, yippee

In [179]:
TxAddr_with_time = TxAddr_edgelist.merge(
    txs_features[["txId", "Time step"]],
    on="txId",
    how="left"
)
print(TxAddr_with_time.head())

        txId                      output_address  Time step
0  230325127  1GASxu5nMntiRKdVtTVRvEbP965G51bhHH          1
1  230325127  14YRXHHof4BY1TVxN5FqYPcEdpmXiYT78a          1
2  230325139  1GFdrdgtG34GChM8SMpMwcXFc4nYbH1A5G          1
3   86875675  19q57SeCEzTnWrWVXA43nZzhSiXkYggh7c          1
4   86875675  1Kk1NVYnCE8ALXDhgMM6HqTt1jDSvi6QBA          1


# Get the prior transaction ones done

In [180]:
output_address_history = TxAddr_with_time.merge(
    address_step_counts[
        ["address", "Time step", "prior_tx_count"]
    ],
    left_on=["output_address", "Time step"],
    right_on=["address", "Time step"],
    how="left"
)
output_address_history = output_address_history.drop(
    columns=["address"]
)
output_prior_tx_stats = (
    output_address_history
    .groupby("txId")["prior_tx_count"]
    .agg(
        output_prior_tx_count="sum",
        output_prior_tx_avg="mean",
        output_prior_tx_max="max"
    )
    .reset_index()
)
print(output_address_history.head())
print(output_prior_tx_stats.head())

        txId                      output_address  Time step  prior_tx_count
0  230325127  1GASxu5nMntiRKdVtTVRvEbP965G51bhHH          1               0
1  230325127  14YRXHHof4BY1TVxN5FqYPcEdpmXiYT78a          1               0
2  230325139  1GFdrdgtG34GChM8SMpMwcXFc4nYbH1A5G          1               0
3   86875675  19q57SeCEzTnWrWVXA43nZzhSiXkYggh7c          1               0
4   86875675  1Kk1NVYnCE8ALXDhgMM6HqTt1jDSvi6QBA          1               0
   txId  output_prior_tx_count  output_prior_tx_avg  output_prior_tx_max
0  1076                      0             0.000000                    0
1  2534                      4             0.571429                    2
2  3181                  12680             2.584590                   16
3  3321                      0             0.000000                    0
4  3889                   1128             1.307068                   21


# Historical BTC send by the output addresses

In [181]:
current_outputs = TxAddr_with_time[
    ["txId", "output_address", "Time step"]
].copy()

current_outputs = current_outputs.rename(columns={
    "txId": "current_txId",
    "Time step": "current_time_step"
})
historical_sends = sending_history[
    ["txId", "input_address", "Time step", "in_BTC_total"]
].copy()

historical_sends = historical_sends.rename(columns={
    "txId": "historical_txId",
    "Time step": "historical_time_step"
})
output_send_matches = current_outputs.merge(
    historical_sends,
    left_on="output_address",
    right_on="input_address",
    how="left"
)
output_send_matches = output_send_matches[
    output_send_matches["historical_time_step"]
    < output_send_matches["current_time_step"]
]
unique_output_prior_sends = output_send_matches.drop_duplicates(
    subset=["current_txId", "historical_txId"]
)
output_btc_sent_stats = (
    unique_output_prior_sends
    .groupby("current_txId")["in_BTC_total"]
    .agg(
        output_btc_sent="sum",
        output_btc_sent_avg="mean",
        output_btc_sent_min="min",
        output_btc_sent_max="max"
    )
    .reset_index()
    .rename(columns={"current_txId": "txId"})
)
print(output_btc_sent_stats.head())

   txId  output_btc_sent  output_btc_sent_avg  output_btc_sent_min  \
0  3181     15141.804171            18.155640             0.001185   
1  3889       443.629829             2.899541             0.001090   
2  5473      2761.376887            57.528685             0.003174   
3  6418      9357.339826            98.498314             3.248224   
4  7952     15604.030378            30.596138             0.001117   

   output_btc_sent_max  
0           773.311466  
1           304.036646  
2           958.374727  
3           956.069135  
4          1006.548526  


In [182]:
receiving_history = TxAddr_with_time.merge(
    txs_features[
        ["txId", "out_BTC_total"]
    ],
    on="txId",
    how="left"
)
historical_receives = receiving_history[
    ["txId", "output_address", "Time step", "out_BTC_total"]
].copy()

historical_receives = historical_receives.rename(columns={
    "txId": "historical_txId",
    "Time step": "historical_time_step"
})
output_receive_matches = current_outputs.merge(
    historical_receives,
    on="output_address",
    how="left"
)
output_receive_matches = output_receive_matches[
    output_receive_matches["historical_time_step"]
    < output_receive_matches["current_time_step"]
]
unique_output_prior_receives = output_receive_matches.drop_duplicates(
    subset=["current_txId", "historical_txId"]
)
output_btc_received_stats = (
    unique_output_prior_receives
    .groupby("current_txId")["out_BTC_total"]
    .agg(
        output_btc_received="sum",
        output_btc_received_avg="mean",
        output_btc_received_min="min",
        output_btc_received_max="max"
    )
    .reset_index()
    .rename(columns={"current_txId": "txId"})
)
print(output_btc_received_stats.head())

   txId  output_btc_received  output_btc_received_avg  \
0  2534            21.222295                 5.305574   
1  3181         20757.216494               105.366581   
2  3889          8616.493146               101.370508   
3  5473          1286.963250                27.382197   
4  6418         12199.091992                63.869592   

   output_btc_received_min  output_btc_received_max  
0                 0.208943                18.175812  
1                 0.018285              3143.184754  
2                 0.079123               885.427232  
3                 0.008767               249.150124  
4                 0.255542               956.066034  


# Now for historical features of the transaction

For now its just fraction of address pairs that have interacted before

In [183]:
address_pairs = AddrTx_with_time.merge(
    TxAddr_with_time[
        ["txId", "output_address"]
    ],
    on="txId",
    how="inner"
)
address_pairs = address_pairs.drop_duplicates(
    subset=["txId", "input_address", "output_address"]
)
pair_first_seen = (
    address_pairs
    .groupby(["input_address", "output_address"])["Time step"]
    .min()
    .reset_index(name="first_seen_step")
)
print(pair_first_seen.head())


                        input_address                      output_address  \
0  111218KKkh1JJFRHbwM16AwCiVCc4m7he1  1A2vTkKSsmVLN2EPEJT3KZR4q1Rvv6c6Xs   
1  111218KKkh1JJFRHbwM16AwCiVCc4m7he1  1KWbPoFkzadegdff9rCK1wBFu3mD8M17Wp   
2   1117wASFaYgJJP6MiY8cPD5DMdQda8gDZ  1K7o3aMfiddvUgMGagdNE5GkiykPPyGj32   
3   1117wASFaYgJJP6MiY8cPD5DMdQda8gDZ  1Po4J4SNyJuGnMGYJfGTXLEvGgAZKiddr7   
4   111HRAJxnoxqyKRVnjqBmwqneUrHc1chi  12RoZAgmZMFHMMrvaqrYZrLMPpAFEFGyWU   

   first_seen_step  
0               17  
1               17  
2                5  
3                5  
4               23  


In [184]:
address_pairs = address_pairs.merge(
    pair_first_seen,
    on=["input_address", "output_address"],
    how="left"
)
address_pairs["seen_before"] = (
    address_pairs["first_seen_step"]
    < address_pairs["Time step"]
)
print(address_pairs.head())

                        input_address       txId  Time step  \
0  14YRXHHof4BY1TVxN5FqYPcEdpmXiYT78a  230325127          1   
1  14YRXHHof4BY1TVxN5FqYPcEdpmXiYT78a  230325127          1   
2  13Lhad3SAmu2vqYg2dxbNcxH7LE77kJu2w  230325139          1   
3  1MAQQZn7EHP6J3erXByCciFiVcgS8ZhWqz   86875675          1   
4  1MAQQZn7EHP6J3erXByCciFiVcgS8ZhWqz   86875675          1   

                       output_address  first_seen_step  seen_before  
0  1GASxu5nMntiRKdVtTVRvEbP965G51bhHH                1        False  
1  14YRXHHof4BY1TVxN5FqYPcEdpmXiYT78a                1        False  
2  1GFdrdgtG34GChM8SMpMwcXFc4nYbH1A5G                1        False  
3  19q57SeCEzTnWrWVXA43nZzhSiXkYggh7c                1        False  
4  1Kk1NVYnCE8ALXDhgMM6HqTt1jDSvi6QBA                1        False  


In [185]:
fraction_pairs_seen_before = (
    address_pairs
    .groupby("txId")["seen_before"]
    .mean()
    .reset_index(
        name="fraction_address_pairs_seen_before"
    )
)
print(fraction_pairs_seen_before.head())

   txId  fraction_address_pairs_seen_before
0  1076                            0.000000
1  2534                            0.000000
2  3181                            0.649817
3  3321                            0.000000
4  3889                            0.000000


# Constructing the csv file!!

In [186]:
final_dataset = txs_features[["txId", "Time step"]].copy()

print(final_dataset.shape)
print(final_dataset.head())

(203769, 2)
    txId  Time step
0   3321          1
1  11108          1
2  51816          1
3  68869          1
4  89273          1


In [187]:
final_dataset = final_dataset.merge(
    input_counts,
    on="txId",
    how="left",
    validate="one_to_one"
)

final_dataset = final_dataset.merge(
    output_counts,
    on="txId",
    how="left",
    validate="one_to_one"
)

In [188]:
current_tx_features = txs_features[
    [
        "txId",
        "in_BTC_total",
        "out_BTC_total",
        "fees"
    ]
].copy()

current_tx_features = current_tx_features.rename(columns={
    "in_BTC_total": "input_total",
    "out_BTC_total": "output_total",
    "fees": "fee"
})

final_dataset = final_dataset.merge(
    current_tx_features,
    on="txId",
    how="left",
    validate="one_to_one"
)



In [189]:

input_prior_tx_count = (
        input_address_history
        .groupby("txId")["prior_tx_count"]
        .sum()
        .reset_index(name="input_prior_tx_count")
    )

final_dataset = final_dataset.merge(
        input_prior_tx_count,
        on="txId",
        how="left",
        validate="one_to_one"
    )

final_dataset = final_dataset.merge(
    input_prior_tx_avg,
    on="txId",
    how="left",
    validate="one_to_one"
)

final_dataset = final_dataset.merge(
    input_prior_tx_max,
    on="txId",
    how="left",
    validate="one_to_one"
)


print(final_dataset.columns)

Index(['txId', 'Time step', 'input_count', 'output_count', 'input_total',
       'output_total', 'fee', 'input_prior_tx_count', 'input_prior_tx_avg',
       'input_prior_tx_max'],
      dtype='str')


In [190]:
final_dataset = final_dataset.merge(
    output_prior_tx_stats,
    on="txId",
    how="left",
    validate="one_to_one"
)
final_dataset = final_dataset.merge(
    output_btc_sent_stats,
    on="txId",
    how="left",
    validate="one_to_one"
)
final_dataset = final_dataset.merge(
    output_btc_received_stats,
    on="txId",
    how="left",
    validate="one_to_one"
)
final_dataset = final_dataset.merge(
    fraction_pairs_seen_before,
    on="txId",
    how="left",
    validate="one_to_one"
)

print(final_dataset.columns)


Index(['txId', 'Time step', 'input_count', 'output_count', 'input_total',
       'output_total', 'fee', 'input_prior_tx_count', 'input_prior_tx_avg',
       'input_prior_tx_max', 'output_prior_tx_count', 'output_prior_tx_avg',
       'output_prior_tx_max', 'output_btc_sent', 'output_btc_sent_avg',
       'output_btc_sent_min', 'output_btc_sent_max', 'output_btc_received',
       'output_btc_received_avg', 'output_btc_received_min',
       'output_btc_received_max', 'fraction_address_pairs_seen_before'],
      dtype='str')


In [195]:
# duplicate_pairs = [
#     ("input_prior_tx_count_x", "input_prior_tx_count_y"),
#     ("input_prior_tx_avg_x", "input_prior_tx_avg_y"),
#     ("input_prior_tx_max_x", "input_prior_tx_max_y"),

#     ("output_prior_tx_count_x", "output_prior_tx_count_y"),
#     ("output_prior_tx_avg_x", "output_prior_tx_avg_y"),
#     ("output_prior_tx_max_x", "output_prior_tx_max_y"),
# ]

# for x, y in duplicate_pairs:
#     print(x, y, final_dataset[x].equals(final_dataset[y]))

In [197]:
# for x, y in duplicate_pairs:
#     base_name = x[:-2]

#     final_dataset[base_name] = final_dataset[x]

#     final_dataset = final_dataset.drop(
#         columns=[x, y]
#     )

In [198]:
print(final_dataset.columns.tolist())

['txId', 'Time step', 'input_count', 'output_count', 'input_total', 'output_total', 'fee', 'input_prior_tx_count', 'input_prior_tx_avg', 'input_prior_tx_max', 'output_prior_tx_count', 'output_prior_tx_avg', 'output_prior_tx_max', 'output_btc_sent', 'output_btc_sent_avg', 'output_btc_sent_min', 'output_btc_sent_max', 'output_btc_received', 'output_btc_received_avg', 'output_btc_received_min', 'output_btc_received_max', 'fraction_address_pairs_seen_before']


In [199]:
final_dataset = final_dataset.merge(
    input_btc_sent,
    on="txId",
    how="left",
    validate="one_to_one"
)

final_dataset = final_dataset.merge(
    input_btc_stats,
    on="txId",
    how="left",
    validate="one_to_one"
)

final_dataset = final_dataset.merge(
    in_out_btc_features,
    on="txId",
    how="left",
    validate="one_to_one"
)

print(final_dataset.columns.tolist())

['txId', 'Time step', 'input_count', 'output_count', 'input_total', 'output_total', 'fee', 'input_prior_tx_count', 'input_prior_tx_avg', 'input_prior_tx_max', 'output_prior_tx_count', 'output_prior_tx_avg', 'output_prior_tx_max', 'output_btc_sent', 'output_btc_sent_avg', 'output_btc_sent_min', 'output_btc_sent_max', 'output_btc_received', 'output_btc_received_avg', 'output_btc_received_min', 'output_btc_received_max', 'fraction_address_pairs_seen_before', 'input_btc_sent', 'input_btc_avg', 'input_btc_min', 'input_btc_max', 'in_BTC_min', 'in_BTC_max', 'in_BTC_mean', 'in_BTC_median', 'in_BTC_total', 'out_BTC_min', 'out_BTC_max', 'out_BTC_mean', 'out_BTC_median', 'out_BTC_total']


In [200]:
final_dataset = final_dataset.merge(
    txs_classes[["txId", "class"]],
    on="txId",
    how="left",
    validate="one_to_one"
)

In [201]:
print(final_dataset["class"].value_counts(dropna=False))

class
3    157205
2     42019
1      4545
Name: count, dtype: int64


In [202]:
final_dataset.to_csv(
    "cmpt496features1.csv",
    index=False
)



In [203]:
print(final_dataset.head())

    txId  Time step  input_count  output_count  input_total  output_total  \
0   3321          1          1.0           2.0     0.534072      0.533972   
1  11108          1          1.0           2.0     5.611878      5.611778   
2  51816          1          1.0           2.0     0.456608      0.456508   
3  68869          1          1.0           2.0     9.308900      9.308800   
4  89273          1          1.0       13107.0   852.164680    852.164680   

      fee  input_prior_tx_count  input_prior_tx_avg  input_prior_tx_max  ...  \
0  0.0001                   0.0                 0.0                 0.0  ...   
1  0.0001                   0.0                 0.0                 0.0  ...   
2  0.0001                   0.0                 0.0                 0.0  ...   
3  0.0001                   0.0                 0.0                 0.0  ...   
4  0.0000                   0.0                 0.0                 0.0  ...   

   in_BTC_max  in_BTC_mean  in_BTC_median  in_BTC_total 

In [204]:

with pd.option_context(
    "display.max_columns", None,
    "display.width", 2000,
    "display.expand_frame_repr", False
):
    print(final_dataset.head(2).to_string(index=False))

 txId  Time step  input_count  output_count  input_total  output_total    fee  input_prior_tx_count  input_prior_tx_avg  input_prior_tx_max  output_prior_tx_count  output_prior_tx_avg  output_prior_tx_max  output_btc_sent  output_btc_sent_avg  output_btc_sent_min  output_btc_sent_max  output_btc_received  output_btc_received_avg  output_btc_received_min  output_btc_received_max  fraction_address_pairs_seen_before  input_btc_sent  input_btc_avg  input_btc_min  input_btc_max  in_BTC_min  in_BTC_max  in_BTC_mean  in_BTC_median  in_BTC_total  out_BTC_min  out_BTC_max  out_BTC_mean  out_BTC_median  out_BTC_total  class
 3321          1          1.0           2.0     0.534072      0.533972 0.0001                   0.0                 0.0                 0.0                    0.0                  0.0                  0.0              NaN                  NaN                  NaN                  NaN                  NaN                      NaN                      NaN                      N

In [209]:
btc_history_cols = [
    "input_btc_sent",
    "input_btc_avg",
    "input_btc_min",
    "input_btc_max",

    "output_btc_sent",
    "output_btc_sent_avg",
    "output_btc_sent_min",
    "output_btc_sent_max",

    "output_btc_received",
    "output_btc_received_avg",
    "output_btc_received_min",
    "output_btc_received_max"
]

final_dataset[btc_history_cols] = (
    final_dataset[btc_history_cols].fillna(0)
)
history_count_cols = [
    "input_prior_tx_count",
    "input_prior_tx_avg",
    "input_prior_tx_max",
    "output_prior_tx_count",
    "output_prior_tx_avg",
    "output_prior_tx_max",
    "fraction_address_pairs_seen_before"
]

final_dataset[history_count_cols] = (
    final_dataset[history_count_cols].fillna(0)
)

In [210]:
print(final_dataset[btc_history_cols].isna().sum())

input_btc_sent             0
input_btc_avg              0
input_btc_min              0
input_btc_max              0
output_btc_sent            0
output_btc_sent_avg        0
output_btc_sent_min        0
output_btc_sent_max        0
output_btc_received        0
output_btc_received_avg    0
output_btc_received_min    0
output_btc_received_max    0
dtype: int64


In [211]:
final_dataset.to_csv(
    "cmpt496features1.csv",
    index=False
)


In [212]:
print(final_dataset.head())

    txId  Time step  input_count  output_count  input_total  output_total  \
0   3321          1          1.0           2.0     0.534072      0.533972   
1  11108          1          1.0           2.0     5.611878      5.611778   
2  51816          1          1.0           2.0     0.456608      0.456508   
3  68869          1          1.0           2.0     9.308900      9.308800   
4  89273          1          1.0       13107.0   852.164680    852.164680   

      fee  input_prior_tx_count  input_prior_tx_avg  input_prior_tx_max  ...  \
0  0.0001                   0.0                 0.0                 0.0  ...   
1  0.0001                   0.0                 0.0                 0.0  ...   
2  0.0001                   0.0                 0.0                 0.0  ...   
3  0.0001                   0.0                 0.0                 0.0  ...   
4  0.0000                   0.0                 0.0                 0.0  ...   

   in_BTC_max  in_BTC_mean  in_BTC_median  in_BTC_total 